# Session-based Binary Prediction with XLNet — CPU

Replicates the Transformers4Rec XLNet architecture using only `transformers==4.30.1` and `torch` (CPU only), modified for binary classification.  
No dependency on transformers4rec or merlin.

In [15]:
import os
import gc
import csv
import glob
import math
import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import LambdaLR

from transformers import XLNetConfig, XLNetModel
from tqdm.auto import tqdm

device = torch.device("cuda")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

Using device: cuda
PyTorch version: 2.5.1+cu121


## Configuration

In [ ]:
# ---------------------------------------------------------------------------
# Feature configuration — sequential (per-position in the session)
# ---------------------------------------------------------------------------
CATEGORICAL_FEATURES = {
    "sess_pid_seq":  {"cardinality": 164386, "embedding_dim": 64},
    "sess_csid_seq": {"cardinality": 622,    "embedding_dim": 32},
    "sess_ccid_seq": {"cardinality": 129,    "embedding_dim": 16},
    "sess_bid_seq":  {"cardinality": 3426,   "embedding_dim": 32},
}

CONTINUOUS_FEATURES = [
    "sess_price_log_norm_seq",
    "sess_dtime_secs_log_norm_seq",
    "sess_prod_recency_days_log_norm_seq",
]

# ---------------------------------------------------------------------------
# Feature configuration — static (one value per session, not sequential)
# Leave empty to disable the static branch entirely.
# ---------------------------------------------------------------------------
STATIC_CATEGORICAL_FEATURES = {
    "user_segment": {"cardinality": 10, "embedding_dim": 8},
    "device_type":  {"cardinality": 5,  "embedding_dim": 4},
}

STATIC_CONTINUOUS_FEATURES = [
    "user_lifetime_days",
    "user_avg_order_value",
]

STATIC_PROJ_DIM = 64  # output dim of the static-feature MLP

# ---------------------------------------------------------------------------
# Optional MLM auxiliary loss (set MASK_PROB=0.0 to disable)
# ---------------------------------------------------------------------------
MASK_PROB = 0.15
MLM_WEIGHT = 0.1
MLM_TARGET_FEATURES = [
    "sess_pid_seq",
    "sess_csid_seq",
]

TARGET_COLUMN = "target"

HAS_STATIC = bool(STATIC_CATEGORICAL_FEATURES) or bool(STATIC_CONTINUOUS_FEATURES)

# ---------------------------------------------------------------------------
# Model hyperparameters
# ---------------------------------------------------------------------------
MAX_SEQ_LEN = 20
D_MODEL = 448
N_HEAD = 8
N_LAYER = 2
CONTINUOUS_PROJ_DIM = 64
CLASSIFIER_HIDDEN_DIM = 128

# ---------------------------------------------------------------------------
# Training hyperparameters
# ---------------------------------------------------------------------------
LEARNING_RATE = 0.00020171456712823088
WEIGHT_DECAY = 2.747484129693843e-05
NUM_EPOCHS = 10
TRAIN_BATCH_SIZE = 256
EVAL_BATCH_SIZE = 512
POS_WEIGHT = 20.0  # weight for positive class in BCE loss (imbalanced data)

# ---------------------------------------------------------------------------
# Run notes
# ---------------------------------------------------------------------------
COMMENT = ""  # free-text notes for this training run

print(f"Target: {TARGET_COLUMN}")
print(f"Static features: {HAS_STATIC}")
print(f"MLM masking: MASK_PROB={MASK_PROB}, targets={MLM_TARGET_FEATURES}")
print(f"POS_WEIGHT: {POS_WEIGHT}")

## Load Data

Provide `train_df` and `test_df` as pandas DataFrames.  
Example: `train_df = pd.read_parquet("train.parquet")` or `train_df = pd.read_pickle("train.pkl")`

In [ ]:
# ---------------------------------------------------------------------------
# Load your data here. The notebook expects two DataFrames:
#   train_df  — training sessions
#   test_df   — test/evaluation sessions
#
# Each row is one session. Required columns:
#   - All keys in CATEGORICAL_FEATURES (list-valued)
#   - All keys in CONTINUOUS_FEATURES (list-valued)
#   - All keys in STATIC_CATEGORICAL_FEATURES (scalar)
#   - All keys in STATIC_CONTINUOUS_FEATURES (scalar)
#   - "sess_seq_len" (int)
#   - TARGET_COLUMN (binary 0/1)
# ---------------------------------------------------------------------------

# train_df = pd.read_parquet("train.parquet")
# test_df = pd.read_parquet("test.parquet")

print(f"Train: {len(train_df)} sessions")
print(f"Test:  {len(test_df)} sessions")
print(f"Columns: {list(train_df.columns)}")

## Dataset

In [ ]:
class SessionDataset(Dataset):
    """Wraps a pandas DataFrame of sessions into a PyTorch Dataset.
    Pads/truncates variable-length sessions to MAX_SEQ_LEN."""

    def __init__(self, df, max_seq_len=MAX_SEQ_LEN):
        self.df = df.reset_index(drop=True)
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        seq_len = min(int(row["sess_seq_len"]), self.max_seq_len)
        result = {}

        # --- Sequential categorical features ---
        for feat_name in CATEGORICAL_FEATURES:
            arr = np.array(row[feat_name], dtype=np.int64)[-self.max_seq_len:]
            padded = np.zeros(self.max_seq_len, dtype=np.int64)
            padded[: len(arr)] = arr
            result[feat_name] = torch.tensor(padded, dtype=torch.long)

        # --- Sequential continuous features ---
        cont_arrays = []
        for feat_name in CONTINUOUS_FEATURES:
            arr = np.array(row[feat_name], dtype=np.float32)[-self.max_seq_len:]
            padded = np.zeros(self.max_seq_len, dtype=np.float32)
            padded[: len(arr)] = arr
            cont_arrays.append(padded)
        result["continuous"] = torch.tensor(
            np.stack(cont_arrays, axis=-1), dtype=torch.float32
        )

        # --- Attention mask ---
        mask = np.zeros(self.max_seq_len, dtype=np.float32)
        mask[:seq_len] = 1.0
        result["attention_mask"] = torch.tensor(mask, dtype=torch.float32)

        # --- Static categorical features (scalars, not sequences) ---
        for feat_name in STATIC_CATEGORICAL_FEATURES:
            result[f"static_{feat_name}"] = torch.tensor(
                int(row[feat_name]), dtype=torch.long
            )

        # --- Static continuous features ---
        if STATIC_CONTINUOUS_FEATURES:
            static_cont = np.array(
                [float(row[f]) for f in STATIC_CONTINUOUS_FEATURES], dtype=np.float32
            )
            result["static_continuous"] = torch.tensor(static_cont, dtype=torch.float32)

        # --- Target ---
        result["target"] = torch.tensor(
            float(row[TARGET_COLUMN]), dtype=torch.float32
        )

        return result


_ds = SessionDataset(train_df)
_item = _ds[0]
print(f"Dataset size: {len(_ds)}")
for k, v in _item.items():
    print(f"  {k}: {v.shape} {v.dtype}")
del _ds, _item

## Model Architecture

In [ ]:
class SessionXLNetModel(nn.Module):
    """XLNet-based session model with a binary classification head.
    Optionally incorporates static features and MLM auxiliary loss."""

    def __init__(self):
        super().__init__()

        # === Sequential feature embeddings ===
        self.embeddings = nn.ModuleDict()
        for feat, cfg in CATEGORICAL_FEATURES.items():
            self.embeddings[feat] = nn.Embedding(
                cfg["cardinality"], cfg["embedding_dim"], padding_idx=0
            )

        self.continuous_proj = nn.Sequential(
            nn.Linear(len(CONTINUOUS_FEATURES), CONTINUOUS_PROJ_DIM),
            nn.ReLU(),
        )

        cat_dim = sum(c["embedding_dim"] for c in CATEGORICAL_FEATURES.values())
        total_input_dim = cat_dim + CONTINUOUS_PROJ_DIM
        self.input_mlp = nn.Sequential(
            nn.Linear(total_input_dim, D_MODEL),
            nn.ReLU(),
        )

        # === XLNet backbone ===
        xlnet_config = XLNetConfig(
            vocab_size=2,
            d_model=D_MODEL,
            n_layer=N_LAYER,
            n_head=N_HEAD,
            d_inner=4 * D_MODEL,
            ff_activation="gelu",
            attn_type="bi",
            mem_len=0,
            dropout=0.0,
        )
        self.xlnet = XLNetModel(xlnet_config)

        # === Static feature branch (optional) ===
        classifier_input_dim = D_MODEL

        if HAS_STATIC:
            self.static_embeddings = nn.ModuleDict()
            for feat, cfg in STATIC_CATEGORICAL_FEATURES.items():
                self.static_embeddings[feat] = nn.Embedding(
                    cfg["cardinality"], cfg["embedding_dim"], padding_idx=0
                )

            static_cat_dim = sum(
                c["embedding_dim"] for c in STATIC_CATEGORICAL_FEATURES.values()
            )
            static_cont_dim = len(STATIC_CONTINUOUS_FEATURES)
            static_raw_dim = static_cat_dim + static_cont_dim

            self.static_mlp = nn.Sequential(
                nn.Linear(static_raw_dim, STATIC_PROJ_DIM),
                nn.ReLU(),
            )

            # LayerNorm on each branch before concat
            self.seq_norm = nn.LayerNorm(D_MODEL)
            self.static_norm = nn.LayerNorm(STATIC_PROJ_DIM)

            classifier_input_dim = D_MODEL + STATIC_PROJ_DIM

        # === MLM auxiliary loss (optional) ===
        self.mlm_heads = nn.ModuleDict()
        if MLM_TARGET_FEATURES:
            self.mask_embedding = nn.Parameter(torch.randn(D_MODEL))
            for feat in MLM_TARGET_FEATURES:
                card = CATEGORICAL_FEATURES[feat]["cardinality"]
                self.mlm_heads[feat] = nn.Linear(D_MODEL, card)
            self.mlm_loss_fn = nn.CrossEntropyLoss()

        # === Classification head ===
        self.classifier = nn.Sequential(
            nn.Linear(classifier_input_dim, CLASSIFIER_HIDDEN_DIM),
            nn.ReLU(),
            nn.Linear(CLASSIFIER_HIDDEN_DIM, 1),
        )

        self.loss_fn = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor([POS_WEIGHT])
        )

    def forward(self, batch):
        # --- Sequential embeddings ---
        cat_embeds = [self.embeddings[f](batch[f]) for f in CATEGORICAL_FEATURES]
        cont_proj = self.continuous_proj(batch["continuous"])

        x = torch.cat(cat_embeds + [cont_proj], dim=-1)
        x = self.input_mlp(x)  # (B, T, D_MODEL)

        attention_mask = batch["attention_mask"]

        # --- Optional MLM masking (training only) ---
        mlm_loss = None
        if self.training and MASK_PROB > 0 and self.mlm_heads:
            # Sample which valid positions to mask
            mask_probs = MASK_PROB * attention_mask  # (B, T), zero out padding
            mlm_mask = torch.bernoulli(mask_probs).bool()  # (B, T)

            if mlm_mask.any():
                # Save original IDs for each target feature
                mlm_targets = {}
                for feat in MLM_TARGET_FEATURES:
                    mlm_targets[feat] = batch[feat][mlm_mask]  # (num_masked,)

                # Replace masked positions with learned mask embedding
                x = x.clone()
                x[mlm_mask] = self.mask_embedding

        # --- XLNet forward ---
        xlnet_out = self.xlnet(inputs_embeds=x, attention_mask=attention_mask)
        hidden = xlnet_out.last_hidden_state  # (B, T, D_MODEL)

        # --- MLM loss computation ---
        if self.training and MASK_PROB > 0 and self.mlm_heads:
            if mlm_mask.any():
                masked_hidden = hidden[mlm_mask]  # (num_masked, D_MODEL)
                feat_losses = []
                for feat in MLM_TARGET_FEATURES:
                    mlm_logits = self.mlm_heads[feat](masked_hidden)
                    feat_losses.append(
                        self.mlm_loss_fn(mlm_logits, mlm_targets[feat])
                    )
                mlm_loss = torch.stack(feat_losses).mean()

        # --- Masked mean pooling ---
        mask_expanded = attention_mask.unsqueeze(-1)  # (B, T, 1)
        hidden_masked = hidden * mask_expanded
        pooled = hidden_masked.sum(dim=1) / mask_expanded.sum(dim=1).clamp(min=1)  # (B, D_MODEL)

        # --- Combine with static features ---
        if HAS_STATIC:
            pooled = self.seq_norm(pooled)

            static_parts = []
            for feat in STATIC_CATEGORICAL_FEATURES:
                static_parts.append(
                    self.static_embeddings[feat](batch[f"static_{feat}"])
                )
            if STATIC_CONTINUOUS_FEATURES:
                static_parts.append(batch["static_continuous"])

            static_cat = torch.cat(static_parts, dim=-1)  # (B, static_raw_dim)
            static_repr = self.static_mlp(static_cat)      # (B, STATIC_PROJ_DIM)
            static_repr = self.static_norm(static_repr)

            pooled = torch.cat([pooled, static_repr], dim=-1)  # (B, D_MODEL + STATIC_PROJ_DIM)

        # --- Classifier ---
        logits = self.classifier(pooled).squeeze(-1)  # (B,)
        targets = batch["target"]
        loss = self.loss_fn(logits, targets)

        # Add MLM auxiliary loss
        if mlm_loss is not None:
            loss = loss + MLM_WEIGHT * mlm_loss

        return loss, logits


model = SessionXLNetModel().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")
if HAS_STATIC:
    n_static = sum(
        p.numel() for n, p in model.named_parameters()
        if "static" in n
    )
    print(f"  Static branch: {n_static:,}")
if model.mlm_heads:
    n_mlm = sum(
        p.numel() for n, p in model.named_parameters()
        if "mlm" in n or "mask_embedding" in n
    )
    print(f"  MLM heads: {n_mlm:,}")

## Evaluation Metrics (Binary Classification)

In [ ]:
from sklearn.metrics import roc_auc_score


def compute_auc(logits, targets):
    """Compute AUC from raw logits and targets. Returns a single float."""
    probs = torch.sigmoid(logits).cpu().numpy()
    targets_np = targets.cpu().numpy()
    try:
        return roc_auc_score(targets_np, probs)
    except ValueError:
        return float("nan")

## Training & Evaluation Helpers

In [ ]:
# ---------------------------------------------------------------------------
# CSV logger
# ---------------------------------------------------------------------------
LOG_FIELDS = [
    "timestamp", "phase", "day", "epoch", "batch", "loss", "auc",
]


def init_log(path):
    with open(path, "w", newline="") as f:
        csv.writer(f).writerow(LOG_FIELDS)
    return path


def log_row(path, **kwargs):
    with open(path, "a", newline="") as f:
        csv.writer(f).writerow([kwargs.get(c, "") for c in LOG_FIELDS])


# ---------------------------------------------------------------------------
# Training (with per-epoch test evaluation)
# ---------------------------------------------------------------------------
def train_model(model, train_loader, test_loader, optimizer, scheduler, log_path):
    """Train for NUM_EPOCHS. After each epoch, evaluate on test_loader.
    Returns (step_losses, epoch_aucs) for plotting."""
    step_losses = []
    epoch_aucs = []

    epoch_bar = tqdm(range(1, NUM_EPOCHS + 1), desc="Epochs", unit="epoch")
    for epoch in epoch_bar:
        model.train()
        epoch_loss = 0.0
        n_batches = 0
        batch_bar = tqdm(train_loader, desc=f"  Epoch {epoch}/{NUM_EPOCHS}",
                         leave=False, unit="batch")
        for batch in batch_bar:
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad()
            loss, _ = model(batch)
            loss.backward()
            optimizer.step()
            scheduler.step()

            batch_loss = loss.item()
            epoch_loss += batch_loss
            n_batches += 1
            step_losses.append(batch_loss)
            batch_bar.set_postfix(loss=f"{batch_loss:.4f}")

            log_row(log_path, timestamp=datetime.datetime.now().isoformat(),
                    phase="train", epoch=epoch,
                    batch=n_batches, loss=f"{batch_loss:.6f}")

        avg_epoch_loss = epoch_loss / max(n_batches, 1)

        # --- End-of-epoch evaluation on test set ---
        test_metrics = evaluate(model, test_loader, log_path=log_path, epoch=epoch)
        test_auc = test_metrics["auc"]
        epoch_aucs.append(test_auc)

        epoch_bar.set_postfix(
            loss=f"{avg_epoch_loss:.4f}",
            test_auc=f"{test_auc:.4f}",
        )
        print(f"    Epoch {epoch}/{NUM_EPOCHS}  "
              f"loss={avg_epoch_loss:.4f}  test_auc={test_auc:.4f}")

    return step_losses, epoch_aucs


# ---------------------------------------------------------------------------
# Evaluation
# ---------------------------------------------------------------------------
def evaluate(model, eval_loader, log_path=None, epoch=None):
    model.eval()
    total_loss = 0.0
    n_batches = 0
    all_logits = []
    all_targets = []

    with torch.no_grad():
        for batch in tqdm(eval_loader, desc="  Evaluating", leave=False, unit="batch"):
            batch = {k: v.to(device) for k, v in batch.items()}
            loss, logits = model(batch)

            total_loss += loss.item()
            n_batches += 1
            all_logits.append(logits.cpu())
            all_targets.append(batch["target"].cpu())

    all_logits = torch.cat(all_logits)
    all_targets = torch.cat(all_targets)
    auc = compute_auc(all_logits, all_targets)
    avg_loss = total_loss / max(n_batches, 1)

    metrics = {"loss": avg_loss, "auc": auc}

    if log_path is not None:
        log_row(log_path,
                timestamp=datetime.datetime.now().isoformat(),
                phase="eval", epoch=epoch,
                loss=f"{avg_loss:.6f}", auc=f"{auc:.6f}")

    return metrics


def wipe_memory():
    gc.collect()

## Train & Evaluate

In [ ]:
# ---------------------------------------------------------------------------
# Build data loaders from DataFrames
# ---------------------------------------------------------------------------
train_ds = SessionDataset(train_df)
test_ds = SessionDataset(test_df)

train_loader = DataLoader(
    train_ds, batch_size=TRAIN_BATCH_SIZE, shuffle=True, drop_last=False,
)
test_loader = DataLoader(
    test_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False,
)

print(f"Train: {len(train_ds)} sessions")
print(f"Test:  {len(test_ds)} sessions")

# ---------------------------------------------------------------------------
# Optimizer & scheduler
# ---------------------------------------------------------------------------
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
total_steps = len(train_loader) * NUM_EPOCHS
scheduler = LambdaLR(
    optimizer,
    lr_lambda=lambda step: max(0.0, 1 - step / max(total_steps, 1)),
)

# ---------------------------------------------------------------------------
# Train (with per-epoch test AUC)
# ---------------------------------------------------------------------------
time_stamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M")
log_path = init_log(f"training_log_{time_stamp}.csv")

print(f"\nTraining for {NUM_EPOCHS} epochs...")
step_losses, epoch_aucs = train_model(
    model, train_loader, test_loader, optimizer, scheduler, log_path,
)

# ---------------------------------------------------------------------------
# Final evaluation summary
# ---------------------------------------------------------------------------
print(f"\nFinal test AUC: {epoch_aucs[-1]:.6f}")
print(f"Best  test AUC: {max(epoch_aucs):.6f} (epoch {epoch_aucs.index(max(epoch_aucs)) + 1})")

# ---------------------------------------------------------------------------
# Save hyperparameters JSON
# ---------------------------------------------------------------------------
import json as _json

hp_record = {
    "timestamp": time_stamp,
    "comment": COMMENT,
    "target_column": TARGET_COLUMN,
    "categorical_features": list(CATEGORICAL_FEATURES.keys()),
    "continuous_features": CONTINUOUS_FEATURES,
    "static_categorical_features": list(STATIC_CATEGORICAL_FEATURES.keys()),
    "static_continuous_features": STATIC_CONTINUOUS_FEATURES,
    "static_proj_dim": STATIC_PROJ_DIM,
    "mask_prob": MASK_PROB,
    "mlm_weight": MLM_WEIGHT,
    "mlm_target_features": MLM_TARGET_FEATURES,
    "max_seq_len": MAX_SEQ_LEN,
    "d_model": D_MODEL,
    "n_head": N_HEAD,
    "n_layer": N_LAYER,
    "continuous_proj_dim": CONTINUOUS_PROJ_DIM,
    "classifier_hidden_dim": CLASSIFIER_HIDDEN_DIM,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "num_epochs": NUM_EPOCHS,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "pos_weight": POS_WEIGHT,
    "train_sessions": len(train_ds),
    "test_sessions": len(test_ds),
    "model_parameters": sum(p.numel() for p in model.parameters()),
    "final_test_auc": epoch_aucs[-1],
    "best_test_auc": max(epoch_aucs),
    "best_epoch": epoch_aucs.index(max(epoch_aucs)) + 1,
    "epoch_aucs": epoch_aucs,
}

hp_path = f"training_{time_stamp}_hp.json"
with open(hp_path, "w") as f:
    _json.dump(hp_record, f, indent=2)
print(f"\nHyperparameters saved to {hp_path}")

## Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# --- Training loss by step ---
ax1.plot(step_losses, linewidth=0.5, alpha=0.7)
ax1.set_xlabel("Step")
ax1.set_ylabel("Loss")
ax1.set_title("Training Loss (by step)")
ax1.grid(True, alpha=0.3)

# --- Test AUC by epoch ---
epochs = list(range(1, len(epoch_aucs) + 1))
ax2.plot(epochs, epoch_aucs, marker="o", linewidth=2)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("AUC")
ax2.set_title("Test AUC (by epoch)")
ax2.set_xticks(epochs)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Save Model

In [ ]:
model_path = os.path.join(os.getcwd(), "saved_model")
os.makedirs(model_path, exist_ok=True)
torch.save(model.state_dict(), os.path.join(model_path, "model.pt"))
print(f"Model saved to {model_path}/model.pt")

## Production-Style Inference

In [ ]:
# ---------------------------------------------------------------------------
# Build tensors for inference (same logic as SessionDataset.__getitem__)
# ---------------------------------------------------------------------------
def df_to_batch(df, max_seq_len=MAX_SEQ_LEN):
    """Convert a DataFrame of sessions into a batched tensor dict."""
    all_cat = {f: [] for f in CATEGORICAL_FEATURES}
    all_cont = []
    all_mask = []

    # Static feature accumulators
    all_static_cat = {f: [] for f in STATIC_CATEGORICAL_FEATURES}
    all_static_cont = []

    for _, row in df.iterrows():
        seq_len = min(int(row["sess_seq_len"]), max_seq_len)

        for feat_name in CATEGORICAL_FEATURES:
            arr = np.array(row[feat_name], dtype=np.int64)[-max_seq_len:]
            padded = np.zeros(max_seq_len, dtype=np.int64)
            padded[:len(arr)] = arr
            all_cat[feat_name].append(padded)

        cont_arrays = []
        for feat_name in CONTINUOUS_FEATURES:
            arr = np.array(row[feat_name], dtype=np.float32)[-max_seq_len:]
            padded = np.zeros(max_seq_len, dtype=np.float32)
            padded[:len(arr)] = arr
            cont_arrays.append(padded)
        all_cont.append(np.stack(cont_arrays, axis=-1))

        mask = np.zeros(max_seq_len, dtype=np.float32)
        mask[:seq_len] = 1.0
        all_mask.append(mask)

        # Static features
        for feat_name in STATIC_CATEGORICAL_FEATURES:
            all_static_cat[feat_name].append(int(row[feat_name]))
        if STATIC_CONTINUOUS_FEATURES:
            all_static_cont.append(
                [float(row[f]) for f in STATIC_CONTINUOUS_FEATURES]
            )

    batch = {}
    for feat_name in CATEGORICAL_FEATURES:
        batch[feat_name] = torch.tensor(np.stack(all_cat[feat_name]), dtype=torch.long)
    batch["continuous"] = torch.tensor(np.stack(all_cont), dtype=torch.float32)
    batch["attention_mask"] = torch.tensor(np.stack(all_mask), dtype=torch.float32)

    # Static features
    for feat_name in STATIC_CATEGORICAL_FEATURES:
        batch[f"static_{feat_name}"] = torch.tensor(
            all_static_cat[feat_name], dtype=torch.long
        )
    if STATIC_CONTINUOUS_FEATURES:
        batch["static_continuous"] = torch.tensor(
            all_static_cont, dtype=torch.float32
        )

    # Dummy target — only needed so forward() doesn't error
    batch["target"] = torch.zeros(len(df), dtype=torch.float32)
    return batch

# ---------------------------------------------------------------------------
# Run inference in mini-batches
# ---------------------------------------------------------------------------
model.eval()
all_logits_list = []

for start in tqdm(range(0, len(test_df), EVAL_BATCH_SIZE), desc="Inference"):
    end = min(start + EVAL_BATCH_SIZE, len(test_df))
    chunk_df = test_df.iloc[start:end]
    batch = df_to_batch(chunk_df)
    batch = {k: v.to(device) for k, v in batch.items()}

    with torch.no_grad():
        _, logits = model(batch)

    all_logits_list.append(logits.cpu().numpy())

all_logits_out = np.concatenate(all_logits_list).astype(np.float32)
all_probs = 1.0 / (1.0 + np.exp(-all_logits_out))  # sigmoid

# ---------------------------------------------------------------------------
# Build final analysis DataFrame
# ---------------------------------------------------------------------------
inference_df = test_df.copy()
inference_df["logit"] = all_logits_out
inference_df["prob"] = all_probs

if TARGET_COLUMN in test_df.columns:
    gt = test_df[TARGET_COLUMN].values.astype(np.float32)
    try:
        auc = roc_auc_score(gt, all_probs)
        print(f"\nAUC: {auc:.6f}")
    except ValueError:
        print("\nAUC: could not compute (single class?)")
    print(f"Positive rate (ground truth): {gt.mean():.4%}")
    print(f"Positive rate (pred > 0.5):   {(all_probs >= 0.5).mean():.4%}")
else:
    print(f"\nNo '{TARGET_COLUMN}' column in test_df — skipping AUC.")

print(f"\nInference DataFrame shape: {inference_df.shape}")
print(inference_df[["sess_seq_len", "logit", "prob"]].head(10))

inference_df.to_pickle("inference_results.pkl")
print(f"\nSaved inference results to inference_results.pkl")